# <center >项目案例：基于DeepSeek-OCR构建AI数据分析系统</center>

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510301750854.png" width=60%></div>

# 一、DeepSeek-OCR vLLM在线服务配置

&emsp;&emsp;在实际的数据分析工作中，我们经常会遇到需要从图片、PDF文档中提取文字信息的场景。传统的OCR工具要么精度不够，要么对中文支持不友好。而 `DeepSeek-OCR` 作为刚刚开源的小型多模态视觉模型，经过我们团队大量的测试，其不仅识别精度高，同时对中文的支持也非常出色，同时也能够输出结构化的 `Markdown` 格式文档，在数据分析领域的数值型处理上表现非常出色。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510211101053.png" width=80%></div>

&emsp;&emsp;首先我们来理解一下什么是OCR。`OCR`（Optical Character Recognition，光学字符识别）简单来说，就是让计算机能够"看懂"图片中的文字。就像我们人眼看到一张照片能够识别出上面写的字一样，OCR技术让计算机也具备了这种能力。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510201841979.png" width=80%></div>

&emsp;&emsp;在实际应用中，OCR技术的使用场景非常广泛：扫描的纸质文档需要转换为可编辑的文本、会议拍照的白板内容需要整理成文档、发票单据需要批量录入系统等等。传统的做法是人工逐字录入，效率低且容易出错。而有了OCR技术，这些工作可以自动化完成。

## 1.1 环境配置要求

&emsp;&emsp;在开始部署之前，我们首先需要确认自己的服务器是否满足运行条件。

<style>
.center 
{
  width: auto;
  display: table;
  margin-left: auto;
  margin-right: auto;
}
</style>

<p align="center"><font face="黑体" size=4>硬件配置要求（建议）</font></p>
<div class="center">

| 硬件类型 | 最低要求 | 推荐配置 | 说明 |
|---------|---------|---------|------|
| GPU | NVIDIA RTX 3090 | RTX 4090 或更高 | 必须是NVIDIA显卡，支持CUDA |
| 显存 | 24GB | 24GB+ | 显存越大，处理速度越快 |
| 内存 | 32GB | 64GB+ | 用于图像预处理和数据缓存 |
| 硬盘 | 50GB可用空间 | 100GB+ | 模型文件较大，需要足够空间 |

</div>



&emsp;&emsp;除了硬件，我们还需要准备相应的软件环境：主要包括**操作系统**：建议使用 `Ubuntu 22.04`，同时需要 `CUDA 12.0+`，而**Python 环境**建议使用 `Python 3.10+`。


&emsp;&emsp;课程使用的配置环境为：`Ubuntu 22.04` + 四卡 3090。大家在使用 vLLM 启动`DeepSeek-OCR`时，首先需要做如下配置校验：

```bash
    # 检查 GPU 是否正常工作
    nvidia-smi
```

&emsp;&emsp;执行上述命令后，如果看到类似下面的输出，说明GPU正常工作：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510301228311.png" width=80%></div>

&emsp;&emsp;接下来检查 Conda 是否安装，执行如下命令：

```bash
    conda --version
```

&emsp;&emsp;如果看到类似 `conda XXX` 的版本号，说明conda已经安装。如果提示命令不存在，需要先安装 miniconda 或者 anaconda3。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510301228312.png" width=80%></div>

## 1.2 DeepSeek-OCR模型下载

&emsp;&emsp;环境准备好之后，接下来我们需要下载 `DeepSeek-OCR` 模型文件，并进行相应的配置。这一步是整个部署过程的核心。国内用户强烈建议大家使用 `ModelScope` 下载模型，没有网络限制，同时速度也会快很多。

&emsp;&emsp;首先，我们需要创建一个独立的 Python 虚拟环境。虚拟环境可以隔离项目依赖，避免与系统其他 Python 项目产生冲突。执行如下命令
```bash
    conda create --name deepseek-ocr-vllm python==3.11
```

&emsp;&emsp;其中：
- `--name deepseek-ocr-vllm`：虚拟环境名称，可以自定义
- `python==3.11`：指定 Python 版本为 3.11（PaddleOCR 推荐版本）

&emsp;&emsp;执行效果如下图所示：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510301257909.png" width=80%></div>

&emsp;&emsp;接下来激活虚拟环境：
```bash
    conda activate deepseek-ocr-vllm
```

&emsp;&emsp;激活后，命令行提示符前会显示 `(deepseek-ocr-vllm)`，表示已进入虚拟环境。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510301257910.png" width=80%></div>

&emsp;&emsp;接下来，在`ModelScope` 平台下载 `DeepSeek-OCR` 模型文件。地址：https://modelscope.cn/models/deepseek-ai/DeepSeek-OCR

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510301300250.png" width=80%></div>

&emsp;&emsp;ModelScope 是阿里云的模型托管平台，国内访问速度快。首先执行如下命令：

```bash
    pip install modelscope
```

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510301301452.png" width=80%></div>

&emsp;&emsp;然后执行如下代码：

```python
    from modelscope import snapshot_download
    model_dir = snapshot_download('deepseek-ai/DeepSeek-OCR',local_dir='.')
```

&emsp;&emsp;其中：
- `'deepseek-ai/DeepSeek-OCR'`：模型仓库 ID
- `local_dir='.'`：下载到当前目录（也可以指定其他路径，如 `'./models'`）

&emsp;&emsp;接下来执行如下代码进行模型权重安装：

```bash
   python download_deepseek_ocr.py
```

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510301304455.png" width=80%></div>

&emsp;&emsp;下载完成后的模型目录结构如下所示：


<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510301323950.png" width=80%></div>

## 1.3 配置模型服务参数

&emsp;&emsp;`DeepSeek-OCR`模型下载完成后，我们需要配置服务的运行参数。项目提供了一个配置模板文件 `.env.example`，我们需要基于它创建自己的配置文件。

&emsp;&emsp;打开 `.env` 文件后，会看到如下配置项：

<style>
.center 
{
  width: auto;
  display: table;
  margin-left: auto;
  margin-right: auto;
}
</style>

<p align="center"><font face="黑体" size=4>核心配置参数说明</font></p>
<div class="center">

| 参数名 | 说明 | 示例值 | 必填/可选 |
|-------|------|--------|----------|
| MODEL_PATH | 模型文件路径 | /home/user/models/deepseek-ocr-1.5b | <font color=red>必填</font> |
| GPU_ID | 使用的GPU编号 | 0 或 1 或 2... | 可选（默认3） |
| PORT | 服务监听端口 | 8708 | 可选（默认8708） |
| HOST | 服务监听地址 | 0.0.0.0 | 可选（默认0.0.0.0） |
| CPU_WORKERS | CPU工作线程数 | 2 | 可选（默认2） |
| CONDA_ENV_NAME | Conda环境名称 | deepseek-ocr-vllm | 二选一 |
| PYTHON_PATH | Python解释器路径 | /path/to/python | 二选一 |

</div>

&emsp;&emsp;其中比较关键的是：**MODEL_PATH（必须修改）** 这是<font color=red>最重要的配置</font>，必须设置为你实际下载模型的路径。

```bash
    MODEL_PATH=/path/to/your/DeepSeek-OCR # 修改成你的实际模型路径
```

&emsp;&emsp;其次，如果你的服务器有多张GPU，需要选择一张来运行模型。建议选择显存大、利用率低的GPU。此外，我们集成了两种方式配置Python环境，<font color=red>选择其中一种即可</font>：

```json
    # 方式1：使用Conda环境名**（适合用conda创建的环境）

    CONDA_ENV_NAME=deepseek-ocr-vllm
    PYTHON_VERSION=3.10
    # PYTHON_PATH=...  # 这一行注释掉

    # 方式2：直接指定Python路径**（适合用miniconda的环境）

    # CONDA_ENV_NAME=...  # 这一行注释掉
    PYTHON_PATH=/home/username/miniconda3/envs/myenv/bin/python
```

> &emsp;如何知道自己的Python路径？执行 `which python` 命令即可查看。

&emsp;&emsp;`PORT` 是服务监听的端口号，默认是 8708。如果这个端口被占用，可以修改为其他值：

```bash
    PORT=8708  # 默认端口
    # 或者换成其他端口
    PORT=9000
```

&emsp;&emsp;`HOST` 控制服务的访问范围：

```bash
    HOST=0.0.0.0  # 允许外部访问（推荐）
    # HOST=127.0.0.1  # 只允许本机访问
```

&emsp;&emsp;如果你希望其他机器能够访问这个OCR服务，<font color=red>必须设置为 0.0.0.0</font>。

## 1.4 启动模型服务

&emsp;&emsp;配置完成后，接下来便可以开始启动服务，让 `DeepSeek-OCR` 真正运行起来。项目提供了两个启动脚本，

- **quick_start.sh（快速启动）**：这是一个简化版本，<font color=red>专为新手设计</font>。它会自动完成环境检查、依赖安装、服务启动等所有步骤，一键到底。适合初次部署或者快速测试。

- **start_server.sh（完整管理）**：这是一个功能完整的交互式脚本，提供了菜单式操作界面。你可以分步骤执行各项操作，还能监控服务状态、查看日志等。适合需要精细控制的场景。

&emsp;&emsp;建议首次部署时使用 `quick_start.sh`，等熟悉流程后再使用 `start_server.sh` 进行日常管理。使用快速启动脚本，整个过程完全自动化，非常简单：

```bash
    # 进入项目目录
    cd /home/MuyuWorkSpace/03_DataAnalysis_main/backend/external/ocr/DeepSeek-OCR-vllm/

    # 给脚本添加执行权限（首次需要）
    chmod +x quick_start.sh

    # 启动服务
    ./quick_start.sh
```

&emsp;&emsp;执行后，你会看到脚本开始工作，输出内容如下：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510301323951.png" width=80%></div>

&emsp;&emsp;`DeepSeek-OCR` 服务提供了自动生成的API文档，可以通过浏览器访问：

```
    http://你的服务器IP:8708/docs
```

&emsp;&emsp;例如，如果是本机，访问：`http://localhost:8708/docs`

&emsp;&emsp;你会看到一个 `Swagger UI` 界面，展示了所有可用的API接口。这个界面不仅可以查看接口文档，还可以直接在网页上测试接口，非常方便。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510301326722.png" width=80%></div>

> &emsp;Swagger 是一个API文档标准，能够自动生成交互式的API文档界面。

## 1.5 Deepseek-OCR源码解读

&emsp;&emsp;`DeepSeek-OCR`通过 vLLM 平台启动，我们是借助`DeepSeek-OCR`官方提供的项目代码，并做了一些优化和调整，其官方源码下载地址：https://github.com/deepseek-ai/DeepSeek-OCR/tree/main/DeepSeek-OCR-master/DeepSeek-OCR-vllm

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510301228310.png" width=80%></div>

&emsp;&emsp;接下来我们深入分析一次OCR请求的完整处理流程。理解这个流程，对于我们后续进行性能优化和问题排查都非常重要。

- **步骤一：文件接收与格式识别**

&emsp;&emsp;当用户上传文件时，服务首先要判断文件类型。我们支持两种输入：单张图片或多页PDF。

```python
    @app.post("/ocr")
    async def ocr(
        file: UploadFile = File(...),           # 上传的文件
        enable_description: bool = Form(False)  # 是否生成图片描述
    ):
        # 读取文件内容
        contents = await file.read()
        
        # 根据文件扩展名判断类型
        if file.filename.lower().endswith('.pdf'):
            # PDF文件：转换为图片列表
            images = await pdf_to_images_async(contents)
        else:
            # 图片文件：直接打开
            images = [await image_open_async(contents)]
```

&emsp;&emsp;这里有一个细节值得注意：<font color=red>所有的IO操作都是异步的</font>。使用 `async/await` 语法，可以让程序在等待IO时不阻塞，去处理其他请求，大幅提升并发能力。

- **步骤二：PDF转图片的优化实现**

&emsp;&emsp;对于PDF文件，我们需要先转换为图片。这个过程比较耗时，因此我们在CPU线程池中执行：

```python
    async def pdf_to_images_async(pdf_bytes: bytes, dpi: int = 144) -> List[Image.Image]:
        """
        异步版本的PDF转图片
        关键：在CPU线程池执行，不阻塞主事件循环
        """
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(cpu_executor, pdf_to_images_sync, pdf_bytes, dpi)

    def pdf_to_images_sync(pdf_bytes: bytes, dpi: int = 144) -> List[Image.Image]:
        """
        同步版本的PDF转图片（实际执行的函数）
        """
        images = []
        doc = fitz.open(stream=pdf_bytes, filetype="pdf")
        matrix = fitz.Matrix(dpi / 72.0, dpi / 72.0)  # DPI 转换矩阵
        
        for page in doc:
            # 渲染页面为图片
            pix = page.get_pixmap(matrix=matrix, alpha=False)
            img = Image.open(io.BytesIO(pix.tobytes("png")))
            
            # 统一转换为RGB格式
            if img.mode != "RGB":
                if img.mode in ("RGBA", "LA"):
                    # 透明背景替换为白色
                    bg = Image.new("RGB", img.size, (255, 255, 255))
                    bg.paste(img, mask=img.split()[-1])
                    img = bg
                else:
                    img = img.convert("RGB")
            
            images.append(img)
        
        doc.close()
        return images
```

&emsp;&emsp;这里的设计思想是：<font color=red>外层异步，内层同步</font>。为什么要这样做？

1. **外层异步**：保证Web服务不阻塞，可以同时处理多个请求；
2. **内层同步**：PDF渲染库本身是同步的，强行改成异步反而会降低性能；
3. **线程池执行**：将同步操作放到独立线程，两全其美；

> &emsp;关于 DPI 参数：默认值144是一个平衡点。DPI越高，图片质量越好，但文件越大、处理越慢。根据实践经验，144 DPI对于大部分文档已经足够。

- **步骤三：批量Tokenize的并发优化**

&emsp;&emsp;这是整个流程中<font color=red>最关键的优化点</font>。Tokenize是指将图片转换为模型可以理解的数字表示，这个过程非常耗时。

```python
    async def vllm_generate_batch_async(
        images: List[Image.Image], 
        prompt: str,
        show_progress: bool = True
    ) -> List[str]:
        """
        批量推理的核心函数
        """
        total = len(images)
        
        # 【优化点1】并发 tokenize
        print(f"   [1/3] Tokenize {total} 页...")
        tokenize_tasks = [tokenize_image_async(img, prompt) for img in images]
        all_tokenized = await asyncio.gather(*tokenize_tasks)
        print(f"   [1/3] Tokenize 完成")
        
        # 【优化点2】构造批量输入
        batch_inputs = [
            {
                "prompt": prompt,
                "multi_modal_data": {"image": tok}
            }
            for tok in all_tokenized
        ]
        
        # 【优化点3】批量GPU推理
        async with vllm_lock:
            print(f"   [2/3] GPU 批量推理 {total} 页...")
            
            def batch_generate():
                # OCR任务的特殊参数
                if prompt == PROMPT_OCR:
                    logits_proc = [NoRepeatNGramLogitsProcessor(20, 50, {128821, 128822})]
                    params = SamplingParams(
                        temperature=0.0,           # 确定性输出
                        max_tokens=2048,           # 最大生成长度
                        skip_special_tokens=False,
                        logits_processors=logits_proc,  # 防止重复
                        repetition_penalty=1.05    # 重复惩罚
                    )
                else:
                    params = SamplingParams(
                        temperature=0.0,
                        max_tokens=512,
                        skip_special_tokens=False
                    )
                
                # 关键：一次性批量推理
                outputs = llm.generate(batch_inputs, params)
                return [out.outputs[0].text for out in outputs]
            
            loop = asyncio.get_event_loop()
            results = await loop.run_in_executor(gpu_executor, batch_generate)
            print(f"   [2/3] GPU 推理完成")
            
            return results
```

&emsp;&emsp;这段代码包含了三个关键优化：

&emsp;&emsp;**优化1：并发Tokenize**

&emsp;&emsp;如果有10张图片，传统做法是逐个处理，总时间 = 10 × 单张时间。而我们使用 `asyncio.gather()` 并发处理，多个图片同时在不同CPU线程中tokenize，总时间 ≈ 单张时间。<font color=red>这里可以节省80%以上的时间</font>。

&emsp;&emsp;**优化2：批量输入构造**

&emsp;&emsp;将所有tokenize结果组织成批量输入格式。这样可以充分利用GPU的并行计算能力。

&emsp;&emsp;**优化3：批量GPU推理**

&emsp;&emsp;一次性将所有图片送入GPU推理。相比逐个推理，批量推理可以：
- 减少GPU调用开销（每次调用都有固定成本）
- 提高GPU利用率（并行计算更多数据）
- 共享中间计算结果（如attention矩阵）

> &emsp;这就像坐电梯，10个人分10次坐和一次坐10个人，显然后者效率更高。GPU推理也是同样的道理。

- **步骤四：后处理与结果清洗**

&emsp;&emsp;模型输出的原始文本包含很多特殊标记，需要清洗处理。这个过程也是并发执行的：

```python
    async def postprocess(idx: int, raw: str, img: Image.Image) -> str:
        """
        后处理单个结果
        """
        # 1. 如果启用了图片描述功能
        if enable_description:
            img_pattern = r'<\|ref\|>image<\|/ref\|><\|det\|>\[\[.*?\]\]<\|/det\|>'
            matches = list(re.finditer(img_pattern, raw))
            
            for match in matches:
                # 生成图片描述
                desc = await generate_image_description_async(img)
                replacement = f"[图片: {desc}]" if desc else "[图片]"
                raw = raw.replace(match.group(0), replacement)
        
        # 2. 清理 Markdown
        cleaned = await clean_markdown_async(raw)
        return cleaned if cleaned else ""

    # 并发处理所有结果
    tasks = [postprocess(i, raw, img) for i, (raw, img) in enumerate(zip(raw_results, images))]
    md_parts = await asyncio.gather(*tasks)

    # 3. 合并最终结果
    final_md = "\n\n".join([md for md in md_parts if md])
```

&emsp;&emsp;Markdown清洗函数会移除模型输出中的特殊标记：

```python
    def clean_markdown_sync(text: str) -> str:
        """清理模型输出的特殊标记"""
        # 移除引用标记
        text = re.sub(r'<\|ref\|>.*?<\|/ref\|>', '', text)
        # 移除检测标记
        text = re.sub(r'<\|det\|>.*?<\|/det\|>', '', text)
        # 移除其他特殊标记
        text = re.sub(r'<\|.*?\|>', '', text)
        # 移除坐标信息
        text = re.sub(r'\[\[.*?\]\]', '', text)
        # 移除分隔线
        text = re.sub(r'={50,}.*?={50,}', '', text, flags=re.DOTALL)
        # 规范化换行
        text = re.sub(r'\n{3,}', '\n\n', text)
        return text.strip()
```

&emsp;&emsp;此外，除了需要了解整体的构建流程，还有如下几个关键的优化技术。这些技术是实现高性能的核心。

- **优化技术一：异步锁机制**

&emsp;&emsp;GPU同时只能处理一个批次的推理任务，如果多个请求同时到达，需要排队处理。我们使用异步锁来实现：

```python
    # 全局异步锁
    vllm_lock = asyncio.Lock()

    async def vllm_generate_async(image: Image.Image, prompt: str) -> str:
        """单张图片推理"""
        # 步骤1: tokenize (不需要锁，可以并发)
        # Tokenize 在 CPU 线程池执行
        # 多个请求并发执行时，每个任务都在独立的线程中执行
        tokenized = await tokenize_image_async(image, prompt)
        
        # 步骤2: GPU推理 (需要锁保护)
        # GPU 是单一共享资源，所有请求共享这个实例
        async with vllm_lock:  # 获取锁，其他协程等待
            # 清理缓存
            loop = asyncio.get_event_loop()
            await loop.run_in_executor(gpu_executor, clear_vllm_cache_sync)
            
            # GPU推理
            result = await loop.run_in_executor(
                gpu_executor,
                vllm_generate_sync,
                tokenized,
                prompt
            )
            
            return result
        # 释放锁，下一个协程可以进入
```

&emsp;&emsp;这个设计的精妙之处在于：<font color=red>tokenize不需要锁，可以多个请求并发执行；只有GPU推理时才加锁排队</font>。这样最大化了并发度，同时保证了GPU的安全使用。

- **优化技术二：缓存管理策略**

&emsp;&emsp;vLLM内部有多模态预处理器缓存，如果不及时清理，会导致显存溢出。我们在每次推理前都清理缓存：

```python
    def clear_vllm_cache_sync():
        """清理 vLLM 缓存"""
        if llm is None:
            return
        try:
            if hasattr(llm.llm_engine, 'input_preprocessor'):
                prep = llm.llm_engine.input_preprocessor
                if hasattr(prep, '_mm_processor_cache'):
                    prep._mm_processor_cache.clear()  # 清空缓存字典
        except:
            pass  # 静默失败，不影响主流程
```

&emsp;&emsp;这是一个防御性编程的典型案例。即使缓存清理失败，也不应该影响主要功能。

# 二、DeepSeek-OCR API 实战应用

&emsp;&emsp;前面我们完成了服务的部署和启动，接下来我们要学习如何在实际项目中调用这个 OCR 服务。本节将通过完整的代码示例，带领大家掌握图片识别和 PDF 文档处理的全流程。

&emsp;&emsp;在开始编写调用代码之前，我们先来了解一下 OCR 服务提供的 API 接口规范。

<style>
.center 
{
  width: auto;
  display: table;
  margin-left: auto;
  margin-right: auto;
}
</style>

<p align="center"><font face="黑体" size=4>OCR API 接口规范</font></p>
<div class="center">

| 项目 | 说明 |
|------|------|
| 接口地址 | `http://localhost:8708/ocr` |
| 请求方式 | POST |
| 内容类型 | multipart/form-data |
| 文件参数 | `file`（支持 jpg, png, pdf 等格式） |
| 可选参数 | `enable_description`（是否生成图片描述，默认 False） |

</div>

&emsp;&emsp;返回的数据格式如下：

```json
  {
      "markdown": "识别出的文本内容（Markdown格式）",
      "page_count": 1,
      "processing_time": 2.34
  }
```

&emsp;&emsp;<font color=red>返回的 `markdown` 字段就是我们需要的识别结果</font>，它保留了原文档的格式结构，包括标题、列表、表格等。


&emsp;&emsp;下面我们通过一个完整的代码示例，来演示如何调用 OCR 服务。

In [1]:
import requests
from pathlib import Path

def ocr_pdf(pdf_path, server_url="http://localhost:8707", save_result=True):
    """
    对 PDF 文档进行 OCR 识别
    
    参数:
        pdf_path: PDF 文件路径
        server_url: OCR 服务地址
        save_result: 是否保存识别结果到文件
        
    返回:
        dict: 包含识别结果的字典
    """
    # 1. 检查文件
    pdf_file = Path(pdf_path)
    if not pdf_file.exists():
        raise FileNotFoundError(f"PDF文件不存在: {pdf_path}")
    
    if pdf_file.suffix.lower() != '.pdf':
        raise ValueError(f"不是 PDF 文件: {pdf_path}")
    
    # 2. 读取文件大小（用于显示）
    file_size_mb = pdf_file.stat().st_size / (1024 * 1024)
    
    print(f"文件名: {pdf_file.name}")
    print(f"文件大小: {file_size_mb:.2f} MB")
    print(f"开始处理...")
    
    # 3. 准备请求
    api_url = f"{server_url}/ocr"
    
    # 4. 发送请求
    with open(pdf_path, 'rb') as f:
        files = {'file': (pdf_file.name, f, 'application/pdf')}
        
        # 这里可以添加额外参数
        # data = {'enable_description': True}  # 启用图片描述（会增加处理时间）
        
        response = requests.post(api_url, files=files)
    
    # 5. 处理结果
    if response.status_code == 200:
        result = response.json()
        
        print(f"处理完成！")
        print(f"统计信息:")
        print(f"   - 总页数: {result['page_count']} 页")
        print(f"   - 处理耗时: {result['processing_time']:.2f} 秒")
        print(f"   - 平均速度: {result['processing_time'] / result['page_count']:.2f} 秒/页")
        
        # 6. 保存结果到文件
        if save_result:
            output_file = pdf_file.with_suffix('.md')
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(result['markdown'])
            print(f"结果已保存到: {output_file}")
        
        return result
    else:
        print(f"处理失败: {response.status_code}")
        print(f"错误信息: {response.text}")
        return None

# 使用示例
# 替换为你的实际 PDF 路径
pdf_file = "./pdfs/10华夏收入混合型证券投资基金2024年年度报告.pdf"

# 调用 OCR 函数
result = ocr_pdf(pdf_file)

if result:
    # 显示部分识别结果
    print("\n" + "="*60)
    print("识别结果预览:")
    print("="*60)
    
    # 显示前 1000 个字符
    preview_text = result['markdown'][:1000]
    print(preview_text)
    
    if len(result['markdown']) > 1000:
        print("\n... (内容过长，已截断)")
        print(f"\n完整内容共 {len(result['markdown'])} 个字符")


文件名: 10华夏收入混合型证券投资基金2024年年度报告.pdf
文件大小: 1.72 MB
开始处理...
处理完成！
统计信息:
   - 总页数: 55 页
   - 处理耗时: 57.91 秒
   - 平均速度: 1.05 秒/页
结果已保存到: pdfs/10华夏收入混合型证券投资基金2024年年度报告.md

识别结果预览:
# 华夏收入混合型证券投资基金  

2024年年度报告  

2024年12月31日

## §1 重要提示及目录  

### 1.1 重要提示  

基金管理人的董事会、董事保证本报告所载资料不存在虚假记载、误导性陈述或重大遗漏，并对其内容的真实性、准确性和完整性承担个别及连带的法律责任。本年度报告已经三分之二以上独立董事签字同意，并由董事长签发。  

基金托管人中国建设银行股份有限公司根据本基金合同规定，于2025年3月27日复核了本报告中的财务指标、净值表现、利润分配情况、财务会计报告、投资组合报告等内容，保证复核内容不存在虚假记载、误导性陈述或者重大遗漏。  

基金管理人承诺以诚实信用、勤勉尽责的原则管理和运用基金资产，但不保证基金一定盈利。  

基金的过往业绩并不代表其未来表现。投资有风险，投资者在作出投资决策前应仔细阅读本基金的招募说明书及其更新。  

本报告中的财务资料经审计，安永华明会计师事务所（特殊普通合伙）为本基金出具了无保留意见的审计报告，请投资者注意阅读。  

本报告期自2024年1月1日起至12月31日止。

### 1.2 目录  

§1 重要提示及目录 1  

§2 基金简介 3  

§3 主要财务指标、基金净值表现及利润分配情况 4  

3.1 主要会计数据和财务指标 4  

3.2 基金净值表现 5  

3.3 过去三年基金的利润分配情况 6  

§4 管理人报告 6  

§5 托管人报告 14  

§6 审计报告 14  

§7 年度财务报表 16  

7.1 资产负债表 17  

7.2 利润表 18  

7.3 净资产变动表 19  

7.4 报表附注 20  

§8 投资组合报告 43  

8.1 期末基金资产组合情况 43  

8.2 报告期末按行业分类的股票投资组合 44  

8.3 期末按公允价值占基金资产净值比例大小排序的所有股票

&emsp;&emsp;通过不同的提示词，对输入的PDF文档的每一页做格式化提取：

```
    # 固定 Prompt
    PROMPT_OCR = "<image>\n<|grounding|>Convert the document to markdown."
    PROMPT_DESC = "<image>\nDescribe this image in detail."
```

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510291231685.png" width=80%></div>

## 三、项目源码解读

&emsp;&emsp;这里我们以一个基金的PDF报告为例，给大家讲解一下实时多模态数据做数据分析的处理流程如下所示：

```json
    1. 用户上传PDF文档
             ↓
    2. OCR识别 + 信息结构化（后台自动完成）
             ↓
    3. 用户提问："分析该基金2024年的整体业绩表现"
             ↓
    4. 调用大模型接口
             ↓
    5. 返回：
        - HTML可视化报告（ECharts图表）
        - 文字摘要
             ↓
    6. 前端显示：
        - 左侧：渲染HTML可视化报告
        - 右侧：聊天框显示摘要文字
             ↓
    7. 用户操作：
    - 点击"全屏查看"：新窗口打开完整报告
```

&emsp;&emsp;DeepSeek-OCR 识别出的 Markdown 内容虽然准确，但对于数据分析来说，还存在几个问题：

<style>
.center 
{
  width: auto;
  display: table;
  margin-left: auto;
  margin-right: auto;
}
</style>

<p align="center"><font face="黑体" size=4>DeepSeek-OCR 原始输出的痛点</font></p>
<div class="center">

| 问题 | 具体表现 | 影响 |
|------|---------|------|
| **长文本难以理解** | 一份50页PDF可能产生10万字符 | LLM上下文长度限制 |
| **数据分散** | 表格、数字、结论混杂在文字中 | 难以快速提取关键信息 |
| **缺乏结构** | 纯文本流，没有层次和分类 | 后续分析效率低 |
| **信息冗余** | 包含大量描述性文字 | 干扰数据分析的准确性 |

</div>




1. <font color=red>**在数据分析领域，很多有效的数据，并不仅仅在表格中，关键的分析结论还可能夹杂在文字中**</font>；
2. <font color=red>**大模型是有上下文限制的，一旦需要同时分析的文档较多，那么是不可能把所有信息都给到大模型**</font>；

&emsp;&emsp;所以，这里就需要将 OCR 识别出的 Markdown 文本进行结构化处理，只保留关键信息供 LLM 分析。

&emsp;&emsp;所以需要将长文档转换为**易于分析的结构化数据**，这里做三步处理流程：

```json
    步骤1：智能切片 (Chunking)
      └─ 按文档标题层级切分
      └─ 控制每块大小在1500 tokens左右
      └─ 保留完整的上下文信息

    步骤2：并发分析 (Concurrent Analysis)
      └─ 对每个块调用 LLM
      └─ 提取：摘要 + 表格 + 关键点
      └─ 多线程加速处理

    步骤3：结构化存储 (Structured Storage)
      └─ 统一 JSON 格式
      └─ 便于后续检索和分析
      └─ 支持可视化和报表生成
```
&emsp;&emsp;核心代码逻辑在：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510301652864.png" width=80%></div>

&emsp;&emsp;最终返回的结构化数据如下所示：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510291402628.png" width=80%></div>
<style>
.center 
{
  width: auto;
  display: table;
  margin-left: auto;
  margin-right: auto;
}
</style>

<p align="center"><font face="黑体" size=4> 方案优化效果对比</font></p>
<div class="center">

| 对比维度 | 直接分析原始Markdown | 结构化预处理 |
|---------|---------------------|-------------|
| **上下文消耗** | 10万字符（接近上限） | 仅传必要信息（~5000字符） |
| **分析准确性** | 容易遗漏关键信息 | 提前提取，准确率高 |
| **处理速度** | 1次LLM调用 | n次切片+1次汇总（但可并发） |
| **成本** | 单次高token消耗 | 分散处理，总成本类似 |
| **可复用性** | 每次都要重新分析 | 结构化数据可缓存复用 |

</div>

&emsp;&emsp;而让LLM 生成可视化报告，我们采取的方案是：<font color=red>**让 LLM 直接生成 HTML + ECharts 可视化代码**</font>

```json
    输入：结构化数据 + 用户问题
            ↓
    【步骤1】构建知识库上下文
      └─ 提取所有表格（最重要）
      └─ 汇总关键要点
      └─ 组织章节摘要
            ↓
    【步骤2】设计 Prompt
      └─ 明确输出格式：HTML + Title + Summary
      └─ 指定可视化需求：至少7张图表
      └─ 定义设计风格：深色霓虹、玻璃拟物
            ↓
    【步骤3】LLM 生成
      └─ 输出完整的 HTML 代码
      └─ 包含 ECharts 可视化
      └─ 附带分析摘要
          ↓
    【步骤4】保存并展示
      └─ 保存为 .html 文件
      └─ 可直接用浏览器打开
      └─ 支持交互和导出
```

&emsp;&emsp;核心代码如下：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510301718354.png" width=80%></div>

&emsp;&emsp;根据得到的 JSON 格式的分析结果，从结构化数据中提取关键信息，构建紧凑的上下文供 LLM 使用。LLM 会生成包含以下内容的交互式报告：

```html
    <!DOCTYPE html>
    <html>
    <head>
        <title>2024年财务分析报告</title>
        <script src="https://cdn.jsdelivr.net/npm/echarts@5/dist/echarts.min.js"></script>
        <style>
            /* 深色霓虹风格 */
            body {
                background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
                color: #fff;
                font-family: Arial, sans-serif;
            }
            /* 三栏布局 */
            .container { display: flex; gap: 20px; }
            .col { flex: 1; }
            /* 玻璃拟物卡片 */
            .panel {
                background: rgba(255,255,255,0.05);
                border: 1px solid rgba(255,255,255,0.2);
                border-radius: 16px;
                padding: 20px;
                margin-bottom: 20px;
            }
        </style>
    </head>
    <body>
        <h1>2024年财务分析报告</h1>
        <div class="container">
            <div class="col" id="colL">
                <!-- 左列：饼图、环图 -->
            </div>
            <div class="col" id="colC">
                <!-- 中列：趋势图、对比图 -->
            </div>
            <div class="col" id="colR">
                <!-- 右列：排行榜、分类柱状图 -->
            </div>
        </div>
        <script>
            // ECharts 初始化代码...
        </script>
    </body>
    </html>
```


<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510291231687.png" width=80%></div>

&emsp;&emsp;将所有步骤整合，形成**从文档到可视化报告的端到端流程**：

```json
    ┌─────────────────────────────────────────────────┐
    │  输入：PDF文档 + 用户问题                        │
    │  "分析2024年财务报告，重点关注收入和利润"        │
    └──────────────────┬──────────────────────────────┘
                    ↓
            【第1步：文档识别】
            DeepSeek-OCR API
                    ↓
            输出：Markdown文本（10万字符）
                    ↓
            【第2步：智能切片】
            按标题分割为30个块
                    ↓
            【第3步：并发结构化提取】
            提取：摘要 + 表格 + 关键点
            耗时：（并发）
                    ↓
            【第4步：知识库构建】
            压缩为紧凑的上下文（2万字符）
                    ↓
            【第5步：LLM生成报告】
            输出：HTML + Title + Summary
                    ↓
            【第6步：保存展示】
            浏览器打开查看交互式报告
                    ↓
    ┌──────────────────┴──────────────────────────────┐
    │  输出：交互式HTML报告 + 文字分析摘要              │
    │  - 7+ 个可视化图表（ECharts）                    │
    │  - 3-5条核心洞察要点                             │
    │  - 支持交互、导出、分享                          │
    └─────────────────────────────────────────────────┘
```

# 三、后端项目启动

&emsp;&emsp;在开始使用后端服务之前，我们需要创建一个Python虚拟环境并安装项目依赖。在项目根目录下执行以下命令：

```bash
    # 进入项目根目录
    cd /path/to/03_DataAnalysis_main

    # 或者使用 conda（如果你使用Anaconda）
    conda create -n ana_ocr python=3.11 -y
```

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510311142051.png" width=80%></div>

&emsp;&emsp;创建虚拟环境后，需要激活虚拟环境：

**Linux/Ubuntu:**
```bash
    conda activate ana_ocr
```
<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510311142052.png" width=80%></div>

&emsp;&emsp;激活成功后，命令行提示符前会出现`(ana_ocr)`字样。项目的所有Python依赖都记录在`backend/requirements.txt`文件中。执行以下命令安装：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510311142053.png" width=80%></div>

```bash
    # 进入后端目录
    cd backend

    # 安装依赖（可能需要几分钟）
    pip install -r requirements.txt

    # 如果下载速度慢，可以使用国内镜像源
    pip install -r requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple
```
<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510311142054.png" width=80%></div>

&emsp;&emsp;如果没有报错，说明依赖安装成功！

&emsp;&emsp;接下来，需要修改 .env 的配置文件（强烈建议结合视频的讲解配置，在本案例的最后结尾部分）

```json
    # DeepSeek-OCR 配置
    DEEPSEEK_MODEL_PATH=/home/data/nongwa/workspace/model/OCR/DeepSeek-OCR
    DEEPSEEK_OCR_URL=http://192.168.110.131:8707/ocr
    DEEPSEEK_OCR_HOST=0.0.0.0
    DEEPSEEK_OCR_PORT=8707

    # 数据分析配置
    DATA_ANALYSIS_BASE_SIZE=1024
    DATA_ANALYSIS_IMAGE_SIZE=640
    DATA_ANALYSIS_CROP_MODE=true
    DATA_ANALYSIS_PROMPT="<image>\n<|grounding|>Convert the document to markdown."

    # 信息结构化配置
    QWEN_TOKENIZER_PATH=/home/MuyuWorkSpace/03_DataAnalysis_main/backend/external/tokenizers/Qwen-tokenizer
    ANALYSIS_CHUNK_SIZE=1500
    ANALYSIS_MAX_WORKERS=10
    ANALYSIS_API_KEY=sk-proj-vFVUlYkgM_dfthtbTE5F17dMT3BlbkFJVaNvuVKwM9FJeO8c8Z_zZGddSCWMqOw0iDd7R9PwSojNqwgjLjhzwqQ_iAZ2f2ytjG1REMdA8A
    ANALYSIS_API_BASE=https://ai.devtool.tech/proxy/v1
    ANALYSIS_MODEL_NAME=gpt-5

    # 可视化配置
    VISUALIZER_API_KEY=sk-proj-vFVUlYkgM_dfthtbqwdmBTF7E5F17dMT3BlbkFJVaNvuVKwM9FJeO8c8Z_zZGddSCWMqOw0iDd7R9PwSojNqwgjLjhzwqQ_iAZ2f2ytjG1REMdA8A
    VISUALIZER_API_BASE=https://ai.devtool.tech/proxy/v1
    VISUALIZER_MODEL_NAME=gpt-5

    # API服务配置
    API_HOST=0.0.0.0
    API_PORT=8708
    API_DEBUG=false
    API_RELOAD=false

    # 文件存储配置
    UPLOAD_DIR=/home/MuyuWorkSpace/03_DataAnalysis_main/backend/outputs/ocr_uploads
    RESULTS_DIR=/home/MuyuWorkSpace/03_DataAnalysis_main/backend/outputs/ocr_results
    TEMP_DIR=/home/MuyuWorkSpace/03_DataAnalysis_main/backend/outputs/ocr_temp

    # 文件处理限制
    MAX_FILE_SIZE_MB=100
    SUPPORTED_EXTENSIONS=.jpg,.jpeg,.png,.pdf

    # 并发和性能配置
    MAX_CONCURRENT_REQUESTS=5
    REQUEST_TIMEOUT=300
    CLEANUP_INTERVAL_HOURS=24

    # 开发环境配置
    ENVIRONMENT=development
    LOG_LEVEL=infoAPI_KEY=sk-e3ffbd43e6e45ef5b
```

&emsp;&emsp;最后，通过一行命令启动后端服务：

```bash
  # 进入后端目录
  cd /backend
  python run.py
```

&emsp;&emsp;启动后，你会看到类似以下输出：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510311158867.png" width=80%></div>

# 四、前端项目启动

&emsp;&emsp;启动前端项目非常简单。在前端目录下执行：(注意：需要确定本地服务器上已经正确配置了 node.js 环境)

```bash
    cd frontend
    npm install
```

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510311135550.png" alt="FuFanManus Architecture" width="800">
</div>

```bash
    npm run dev
```

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510311135551.png" alt="FuFanManus Architecture" width="800">
</div>

&emsp;&emsp;稍等片刻，你会看到类似这样的输出：

```
    VITE v6.3.7  ready in 250 ms

    ➜  Local:   http://localhost:3000/
    ➜  Network: http://192.168.110.131:3000/
    ➜  press h + enter to show help
```

&emsp;&emsp;这表示开发服务器已成功启动！注意这几个关键信息：

- **Local**: `http://localhost:3000/` - 本地访问地址（只能在本机访问）
- **Network**: `http://192.168.110.131:3000/` - 网络访问地址（局域网内其他设备可访问）

&emsp;&emsp;在浏览器中打开 `http://localhost:3000`，你应该能看到系统的登录界面或首页。

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510311137795.png" alt="FuFanManus Architecture" width="800">
</div>


&emsp;&emsp;完整启动前后端项目后，即可在网页中上传PDF多模态文档，并进行可视化数据分析问答。

# 附录：Windows 安装 Node.js

&emsp;&emsp;`Node.js` 是一个基于 `Chrome V8` 引擎的 `JavaScript` 运行时环境，广泛应用于前端开发、后端服务器开发等场景。接下来我们详细介绍如何在 `Windows` 系统上从零开始安装和配置 `Node.js`，包括环境变量设置、`npm` 配置等关键步骤。

- **Step 1：下载 Node.js 安装包**

&emsp;&emsp;首先需要从官方网站下载适合你系统的 `Node.js` 安装包。`Node.js` 提供了稳定的 `LTS`（长期支持）版本和最新的 `Current` 版本，推荐选择 `LTS` 版本以确保稳定性。访问 `Node.js` 官网下载页：
   - 官方地址：[https://nodejs.org/en/download/](https://nodejs.org/en/download/)
   - 中文镜像：[https://nodejs.cn/download/](https://nodejs.cn/en/download)

&emsp;&emsp;优先选择 **LTS（长期支持）版本**，稳定性更好, 选择 `.msi` 安装包（根据系统架构选择 64-bit 或 32-bit）

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202509181529316.png" alt="FuFanManus Architecture" width="800">
</div>

&emsp;&emsp;**点击下载**对应的 `.msi` 安装程序即可。

- **Step 2：运行安装程序**

&emsp;&emsp;下载完成后，需要运行 `.msi` 安装程序进行安装。安装过程中需要注意安装路径的选择以及环境变量的配置。首先找到下载的 `.msi` 文件，双击运行：

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202509181535199.png" alt="FuFanManus Architecture" width="800">
</div>

&emsp;&emsp;安装向导出现后，点击 **Next / 下一步** 继续，阅读 `License Agreement`，勾选 "I accept the terms in the License Agreement"，点击 **Next**：

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202509181535201.png" alt="FuFanManus Architecture" width="800">
</div>

&emsp;&emsp;这里强烈建议选择默认的路径：`C:\Program Files\nodejs\`，如必须修改，注意：路径中不要包含中文字符或特殊符号。

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202509181535202.png" alt="FuFanManus Architecture" width="800">
</div>

&emsp;&emsp;自定义设置， 点击 "Install" 开始安装，系统可能会弹出权限确认，选择"是"或"允许"
   - 勾选 "Add to PATH" 选项
   - 勾选 "Automatically install npm package manager"
   - 这样可以在任何命令行窗口中直接使用 `node` 和 `npm` 命令

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202509181535203.png" alt="FuFanManus Architecture" width="800">
</div>

&emsp;&emsp;等待安装完成即可。

# 附录：在 Ubuntu 系统上安装 Node.js

&emsp;&emsp;在实际的生产环境和服务器部署中，`Linux`系统（尤其是`Ubuntu`）是最常见的选择。相比`Windows`，`Linux`系统更加轻量、稳定，且对开发者友好。因此，掌握在`Ubuntu`上安装和配置`Node.js`是每一位全栈开发者的必备技能。

&emsp;&emsp;接下来，我将为大家详细讲解在`Ubuntu`系统上安装`Node.js`的方法。首先，我们需要更新系统的软件包列表，确保获取最新的软件信息：

```bash
    # 更新软件包索引
    sudo apt update

    # 升级已安装的软件包（可选）
    sudo apt upgrade -y
```

&emsp;&emsp;这个命令会联网下载最新的软件包列表，确保我们安装的是最新可用版本。接下来安装 Node.js 和 npm

```bash
    # 一条命令同时安装 Node.js 和 npm
    sudo apt install nodejs npm -y
```

&emsp;&emsp;等待安装完成后，验证安装：

```bash
    node -v    # 显示 Node.js 版本（可能是 v12 或 v14）
    npm -v     # 显示 npm 版本
```

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510231332604.png" alt="FuFanManus Architecture" width="800">
</div>